<a href="https://colab.research.google.com/github/racoope70/daytrading-with-ml/blob/main/lightgbm_enhanced_strategy_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get remove --purge -y cuda* libcuda* nvidia* || echo "No conflicting CUDA packages"
!apt-get autoremove -y
!apt-get clean

In [2]:
#Protocol Buffer Fix (for TensorFlow)
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

In [3]:
#Update Colab Environment and System Libraries
!apt-get update -y && apt-get upgrade -y


In [4]:
#Install Correct Version of CUDA for Colab GPU
!apt-get update -qq && apt-get install -y \
    libcusolver11 libcusparse11 libcurand10 libcufft10 libnppig10 libnppc10 libnppial10 \
    cuda-toolkit-12-4

In [5]:
#Set Correct CUDA Paths
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda-12.4'
os.environ['PATH'] += ':/usr/local/cuda-12.4/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-12.4/lib64'


In [6]:
#Install RAPIDS and NVIDIA Dependencies
!pip install --extra-index-url=https://pypi.nvidia.com \
    cuml-cu12==25.2.0 cudf-cu12==25.2.0 cupy-cuda12x dask-cuda==25.2.0 dask-cudf-cu12==25.2.0


In [7]:
#Install TensorFlow (latest GPU-compatible version)
!pip install tensorflow==2.18.0

#Install Stable Baselines3 and Trading Libraries
!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance xgboost joblib

#Install Miscellaneous Libraries
!pip install matplotlib scikit-learn pandas numba==0.61.0

#Install PyTorch with GPU Support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


In [8]:
#Install TensorFlow (latest GPU-compatible version)
!pip install tensorflow==2.18.0


import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU memory growth enabled")
    except RuntimeError as e:
        print(f"TensorFlow GPU memory config failed: {e}")


In [9]:
!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance --quiet
!pip install stable-baselines3[extra] --quiet


In [10]:
!rm -rf /content/drive

In [12]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

In [13]:
#Add Live Updating / Online Learning
!pip install yfinance lightgbm scikit-learn matplotlib


In [14]:
#Noise Filtering
!pip install PyWavelets

In [30]:
pip install -U scikit-learn


In [8]:
# === Imports and Warning Fixes ===
import os, gc, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import pywt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

# Fix FutureWarning for 'ensure_all_finite'
warnings.filterwarnings("ignore", message=".*ensure_all_finite.*", category=FutureWarning)

# === Config ===
TEST_MODE = False
TICKERS = ['AAPL'] if TEST_MODE else [
    'AAPL', 'TSLA', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK-B', 'JPM', 'JNJ',
    'XOM', 'V', 'PG', 'UNH', 'MA', 'HD', 'LLY', 'MRK', 'PEP', 'KO', 'BAC', 'ABBV',
    'AVGO', 'COST', 'DIS', 'WMT', 'PFE', 'CSCO', 'TMO', 'ACN', 'CVX', 'ABT', 'CRM',
    'MCD', 'INTC', 'DHR', 'QCOM', 'TXN', 'NEE', 'AMGN', 'LIN', 'PM', 'BMY', 'MDT',
    'HON', 'UNP', 'LOW', 'ORCL', 'NKE', 'RTX', 'MS', 'GS'
]

PERIOD = "720d"
INTERVAL = "1h"
SAVE_DIR = "/content/drive/MyDrive/QuantConnect/results_lightgbm/lightgbm_walkforward_models"
RESULTS_DIR = "/content/drive/MyDrive/QuantConnect/results_lightgbm"
OUTPUT_PATH = f"{RESULTS_DIR}/multi_stock_feature_engineered_dataset.csv"
SLIPPAGE_RATE = 0.001
THRESHOLD = 0.55
STOP_LOSS_PCT = 0.03
FEATURES = ["SMA_50", "EMA_20", "RSI_voladj", "MACD_voladj", "Signal_Line", "ATR_voladj", "OBV", "CCI_voladj"]

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "data"), exist_ok=True)

# === Noise Filtering ===
def wavelet_denoise(series, wavelet="db1"):
    coeffs = pywt.wavedec(series, wavelet, mode="symmetric")
    coeffs[1:] = [np.zeros_like(c) for c in coeffs[1:]]
    return pywt.waverec(coeffs, wavelet, mode="symmetric")[:len(series)]

# === Market Regime Detection ===
def detect_regime(df, window=100):
    df["Return"] = df["Close"].pct_change()
    df["Vol"] = df["Close"].rolling(window).std()
    valid = df[["Return", "Vol"]].dropna()
    if len(valid) < 50:
        df["Regime"] = 1
        return df
    model = KMeans(n_clusters=3, random_state=42)
    labels = model.fit_predict(valid)
    regime = pd.Series(index=valid.index, data=labels)
    df["Regime"] = regime.reindex(df.index).ffill().bfill()
    return df

# === Feature Engineering ===
def compute_features(df):
    df['SMA_50'] = df['Close'].rolling(50).mean()
    df['EMA_20'] = df['Close'].ewm(span=20).mean()
    delta = df['Close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-6)
    df['RSI'] = 100 - (100 / (1 + rs))
    df['MACD'] = df['Close'].ewm(12).mean() - df['Close'].ewm(26).mean()
    df['Signal_Line'] = df['MACD'].ewm(9).mean()
    df['ATR'] = df['High'].rolling(14).max() - df['Low'].rolling(14).min()
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    tp = (df['High'] + df['Low'] + df['Close']) / 3
    df['CCI'] = (tp - tp.rolling(20).mean()) / (0.015 * tp.rolling(20).std())
    for col in ["RSI", "MACD", "CCI", "OBV", "ATR"]:
        df[f"{col}_voladj"] = wavelet_denoise(df[col].fillna(0).values)
    df = detect_regime(df)
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df

# === Robust Downloader ===
def download_data(ticker, retries=3):
    for attempt in range(retries):
        try:
            df = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)
            if not df.empty:
                df.reset_index(inplace=True)
                df['Symbol'] = ticker
                return df
        except Exception as e:
            print(f"Retry {attempt+1} for {ticker}: {e}")
            time.sleep((attempt + 1) * 5)
    print(f"Failed: {ticker}")
    return None

# === Save Preprocessed Dataset ===
all_data = []
for ticker in TICKERS:
    print(f"Processing {ticker}")
    df_raw = download_data(ticker)
    if df_raw is None:
        continue
    try:
        df_feat = compute_features(df_raw)
        all_data.append(df_feat)
    except Exception as e:
        print(f"Feature failure: {ticker} -> {e}")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved to {OUTPUT_PATH} | Shape: {final_df.shape}")
else:
    print("No data processed")

# === Walkforward LGBM Training & Evaluation ===
def run_lgbm_walkforward(ticker, window_size=2000, step_size=250, initial_cash=100000):
    print(f"\n{ticker} | LGBM Walkforward")
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL, auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
    df = compute_features(df)
    if df.empty or not all(col in df.columns for col in FEATURES): return

    scaler = MinMaxScaler()
    capital = initial_cash
    shares = capital / df['Close'].iloc[0]
    model = LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42, force_row_wise=True)
    port_lgbm, port_hold, y_preds, y_true = [], [], [], []

    for start in range(0, len(df) - window_size, step_size):
        segment = df.iloc[start:start + window_size].copy()
        if segment.shape[0] < window_size: break
        X = scaler.fit_transform(segment[FEATURES])
        y = segment['Target']
        X_train, X_test = X[:-step_size], X[-step_size:]
        y_train, y_test = y[:-step_size], y[-step_size:]
        prices = segment['Close'].values[-step_size:]

        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  callbacks=[early_stopping(20), log_evaluation(0)])

        probs = model.predict_proba(X_test)[:, 1]
        preds = (probs > THRESHOLD).astype(int)
        y_preds.extend(preds)
        y_true.extend(y_test.tolist())

        for i, pred in enumerate(preds):
            raw_price = prices[i]
            exec_price = raw_price * (1 + SLIPPAGE_RATE) if pred else raw_price * (1 - SLIPPAGE_RATE)
            if pred:
                if raw_price < exec_price * (1 - STOP_LOSS_PCT):
                    capital = shares * raw_price
                else:
                    capital = shares * exec_price
            else:
                shares = capital / exec_price
            port_lgbm.append(capital)
            port_hold.append(shares * raw_price)
        time.sleep(0.01)

    if len(y_preds) < 2: return
    y_preds, y_true = np.array(y_preds), np.array(y_true[:len(y_preds)])
    returns = np.diff(port_lgbm) / (np.array(port_lgbm[:-1]) + 1e-6)
    sharpe = np.mean(returns) / (np.std(returns) + 1e-6) * np.sqrt(252)
    drawdown = np.max(np.maximum.accumulate(port_lgbm) - port_lgbm)

    metrics = {
        "Ticker": ticker,
        "Final_Portfolio": round(port_lgbm[-1], 2),
        "Return_%": round((port_lgbm[-1] - initial_cash) / initial_cash * 100, 2),
        "Sharpe": round(sharpe, 4),
        "Accuracy": round(accuracy_score(y_true, y_preds), 4),
        "Precision": round(precision_score(y_true, y_preds), 4),
        "Recall": round(recall_score(y_true, y_preds), 4),
        "F1_Score": round(f1_score(y_true, y_preds), 4),
        "Drawdown": round(drawdown, 2)
    }

    pd.DataFrame([metrics]).to_csv(f"{RESULTS_DIR}/metrics_{ticker}.csv", index=False)
    model.booster_.save_model(f"{SAVE_DIR}/model_{ticker}.txt")

    plt.figure(figsize=(12, 6))
    plt.plot(port_lgbm, label="LGBM Strategy")
    plt.plot(port_hold, label="Buy & Hold")
    plt.title(f"{ticker} Portfolio Value")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/plots/{ticker}_portfolio_plot.png")
    plt.close()
    print(f"{ticker}: Return {metrics['Return_%']}%, Sharpe {metrics['Sharpe']}")

    summary_path = os.path.join(RESULTS_DIR, "lightgbm_walkforward_summary.csv")
    if os.path.exists(summary_path):
        all_df = pd.read_csv(summary_path)
        all_df = all_df[all_df["Ticker"] != ticker]
        all_df = pd.concat([all_df, pd.DataFrame([metrics])], ignore_index=True)
    else:
        all_df = pd.DataFrame([metrics])
    all_df.to_csv(summary_path, index=False)

# === Run All Tickers ===
for ticker in TICKERS:
    run_lgbm_walkforward(ticker)
    gc.collect()
    time.sleep(1)


Saved to /content/drive/MyDrive/QuantConnect/results_lightgbm/multi_stock_feature_engineered_dataset.csv | Shape: (255683, 279)

AAPL: Return 31.81%, Sharpe 0.2973
TSLA: Return -19.82%, Sharpe -0.2552
MSFT: Return 1.32%, Sharpe 0.0771
GOOGL: Return 1.34%, Sharpe 0.0636
AMZN: Return -7.74%, Sharpe -0.1292
NVDA: Return 66.37%, Sharpe 0.5882
META: Return -10.29%, Sharpe -0.158
BRK-B: Return 16.59%, Sharpe 0.6302
JPM: Return -12.85%, Sharpe -0.3378
JNJ: Return -6.26%, Sharpe -0.2085
XOM: Return 3.29%, Sharpe 0.0845
V: Return 16.27%, Sharpe 0.3724
PG: Return 11.84%, Sharpe 0.3633
UNH: Return -9.51%, Sharpe -0.3949
MA: Return 12.69%, Sharpe 0.4927
HD: Return 9.49%, Sharpe 0.8113
LLY: Return 2.88%, Sharpe 0.1515
MRK: Return 0.0%, Sharpe 0.0
PEP: Return 9.43%, Sharpe 0.4998
KO: Return 6.3%, Sharpe 0.3871
BAC: Return -0.09%, Sharpe 0.0039
ABBV: Return -5.8%, Sharpe -0.3993
AVGO: Return -4.22%, Sharpe -0.1252
COST: Return -4.21%, Sharpe -0.1943
DIS: Return 11.81%, Sharpe 0.4164
WMT: Return 31.11